In [ ]:
# Imports and installations
!pip install -q transformers accelerate qwen-vl-utils pillow

import os
import json
import cv2
import torch

from PIL import Image
from tqdm import tqdm
from google.colab import drive
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

drive.mount("/content/drive")


In [2]:
# Paths
BASE = "/content/drive/MyDrive/HDA-IWBF-2023/synthetic-tattoo-images"

IMAGE_DIR = os.path.join(BASE, "JPGImages")

FEATURES_JSON_PATH = os.path.join(
    BASE,
    "features_merged",
    "features_all.json"
)

REPORTS_OUT_DIR = os.path.join(BASE, "forensic_reports_qwen_batches")
os.makedirs(REPORTS_OUT_DIR, exist_ok=True)

In [14]:
# Batch config
REPORT_BATCH_SIZE = 25
REPORT_BATCH_INDEX = 4

MAX_NEW_TOKENS = 700
MARGIN_RATIO = 0.08

In [15]:
# Output paths
BATCH_REPORTS_DIR = os.path.join(
    REPORTS_OUT_DIR,
    f"reports_batch_{REPORT_BATCH_INDEX:02d}"
)

os.makedirs(BATCH_REPORTS_DIR, exist_ok=True)

BATCH_ERRORS_OUT = os.path.join(
    REPORTS_OUT_DIR,
    f"errors_reports_batch_{REPORT_BATCH_INDEX:02d}.json"
)

BATCH_SUMMARY_OUT = os.path.join(
    REPORTS_OUT_DIR,
    f"summary_reports_batch_{REPORT_BATCH_INDEX:02d}.json"
)

In [ ]:
# Model loading
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

print("Model loaded correctly.")

In [16]:
# Helpers
def read_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def read_image_bgr(path: str):
    img = cv2.imread(path, cv2.IMREAD_COLOR)

    if img is None:
        raise FileNotFoundError(f"Could not read image: {path}")

    return img


def expand_bbox(x, y, w, h, img_shape, margin_ratio=0.08):
    H, W = img_shape[:2]

    mx = int(w * margin_ratio)
    my = int(h * margin_ratio)

    x0 = max(0, x - mx)
    y0 = max(0, y - my)
    x1 = min(W - 1, x + w - 1 + mx)
    y1 = min(H - 1, y + h - 1 + my)

    return x0, y0, x1, y1


def crop_from_features(img_bgr, feat_json, margin_ratio=0.08):
    bbox = feat_json["spatial_features"]["bbox"]

    if bbox is None:
        raise ValueError("This sample has no valid tattoo bounding box.")

    x = int(bbox["x"])
    y = int(bbox["y"])
    w = int(bbox["w"])
    h = int(bbox["h"])

    x0, y0, x1, y1 = expand_bbox(
        x, y, w, h,
        img_bgr.shape,
        margin_ratio=margin_ratio
    )

    crop_bgr = img_bgr[y0:y1 + 1, x0:x1 + 1].copy()

    return crop_bgr


def bgr_to_pil(img_bgr):
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    return Image.fromarray(rgb)


def size_label(tattoo_area_frac_img):
    if tattoo_area_frac_img < 0.015:
        return "small relative to the full image"
    elif tattoo_area_frac_img < 0.06:
        return "medium-sized relative to the full image"
    else:
        return "large relative to the full image"


def skin_coverage_label(tattoo_over_skin_frac):
    if tattoo_over_skin_frac < 0.04:
        return "a small portion of the visible skin region"
    elif tattoo_over_skin_frac < 0.12:
        return "a moderate portion of the visible skin region"
    else:
        return "a large portion of the visible skin region"


def aspect_label(aspect_ratio):
    if aspect_ratio < 0.65:
        return "predominantly vertical and taller than wide"
    elif aspect_ratio > 1.45:
        return "predominantly horizontal and wider than tall"
    else:
        return "approximately balanced in width and height"


def circularity_label(circularity):
    if circularity < 0.35:
        return "irregular and non-circular"
    elif circularity < 0.65:
        return "moderately irregular"
    else:
        return "relatively compact or rounded"


def solidity_label(solidity):
    if solidity < 0.70:
        return "visually fragmented or containing substantial concavities"
    elif solidity < 0.90:
        return "moderately compact with some irregularity"
    else:
        return "compact and mostly solid in outline"


def elongation_label(elongation):
    if elongation < 1.4:
        return "not strongly elongated"
    elif elongation < 2.5:
        return "moderately elongated"
    else:
        return "strongly elongated"


def contrast_label(cohens_d):
    if cohens_d is None:
        return "contrast against the surrounding skin could not be reliably estimated"

    if cohens_d < 0.5:
        return "low contrast against the surrounding skin"
    elif cohens_d < 1.2:
        return "moderate contrast against the surrounding skin"
    else:
        return "strong contrast against the surrounding skin"


def edge_density_label(edge_density):
    if edge_density is None:
        return "edge density could not be reliably estimated"

    if edge_density < 0.015:
        return "low detected edge density"
    elif edge_density < 0.05:
        return "moderate detected edge density"
    else:
        return "high detected edge density"


def entropy_label(entropy):
    if entropy is None:
        return "tonal complexity could not be reliably estimated"

    if entropy < 5.5:
        return "low tonal variation"
    elif entropy < 7.0:
        return "moderate tonal variation"
    else:
        return "high tonal variation"

def no_tattoo_detected(feat_json):
    return feat_json["spatial_features"]["tattoo_area_px"] == 0

In [17]:
# Prompt Builder

def build_forensic_prompt(feat_json: dict) -> str:
    spatial = feat_json["spatial_features"]
    shape = feat_json["shape_features"]
    appearance = feat_json["appearance_features"]

    prompt = f"""
You are a forensic tattoo description assistant.

All responses must be written strictly in English.
Use a neutral, professional, evidence-based forensic tone.

You will receive:
1. a cropped image focused on the tattoo region
2. structured spatial, morphological, and appearance information extracted automatically from image analysis

Your task is to generate a careful forensic-style description of the tattoo.

VERY IMPORTANT:
- The system only receives a tattoo-centered crop.
- The model does NOT have access to full-body anatomical context.
- Never infer or guess anatomical body location.
- Never use words such as arm, leg, chest, shoulder, wrist, thigh, torso, neck, hand, calf, or back.
- Spatial position refers ONLY to relative image coordinates inside the original image.
- The provided spatial position labels are NOT anatomical labels.

STRICT RULES:
- Describe ONLY directly visible visual properties.
- Be conservative and evidence-grounded.
- Do NOT invent information that is not clearly supported by the image.
- Do NOT infer identity, ethnicity, nationality, religion, ideology, gang affiliation, profession, emotional state, personality, or personal history.
- Do NOT speculate about symbolism, meaning, intention, or cultural significance.
- If uncertainty exists, explicitly acknowledge uncertainty.
- Prioritize precision over creativity.
- Do NOT mention raw numeric feature values.
- Translate structured evidence into qualitative forensic language.
- Do NOT generate a generic image caption.
- The structured evidence must explicitly influence the reasoning.
- Treat edge density only as contour activity or structural variation, NOT as artistic quality.
- Avoid repetitive wording across sections.
- Avoid exaggerated claims such as "unique", "impossible to replicate", "reliable identifier", "clear identifier", "highly distinctive", or "perfect identifier".
- Avoid unsupported claims of symmetry. Only mention symmetry if it is clearly visible.
- Avoid saying there are "no irregularities" or "no fragmentation" unless clearly supported by both image and structured evidence.
- If visual evidence and structured evidence seem inconsistent, prioritize cautious visual interpretation.

Structured evidence already interpreted qualitatively:

[Spatial Interpretation]
- The tattoo is {size_label(spatial["tattoo_area_frac_img"])}.
- It occupies {skin_coverage_label(spatial["tattoo_over_skin_frac"])}.
- Its relative spatial position inside the original image corresponds to the {spatial["position"]["label"]} region.
- Its overall geometric proportion is {aspect_label(spatial["bbox_aspect_ratio"])}.

[Shape Interpretation]
- The external contour appears {circularity_label(shape["circularity"])}.
- The overall structure appears {solidity_label(shape["solidity"])}.
- The global geometry appears {elongation_label(shape["elongation"])}.

[Appearance Interpretation]
- The tattoo shows {contrast_label(appearance["cohens_d"])}.
- The crop presents {edge_density_label(appearance["edge_density_crop"])}.
- The image region shows {entropy_label(appearance["entropy_crop"])}.

Return EXACTLY the following sections and titles:

[Observed Description]
Write 3-4 concise sentences.
Describe:
- apparent motif or visual subject
- visible composition
- overall visual organization
- approximate relative position within the image
- whether the tattoo appears line-based, shaded, filled, ornamental, geometric, realistic, stylized, etc.
Describe only directly visible properties.

[Feature-Based Interpretation]
Write 5-7 sentences.
This is the most important section for explainability.
Explain how the structured evidence supports the visual description.
Use qualitative forensic language only.
Do NOT mention:
- numeric values
- raw measurements
- mathematical terminology
Focus on:
- relative size
- relative position within the image
- geometric behavior
- compactness
- contour regularity
- elongation
- tonal behavior
- contrast behavior
- structural appearance
Make clear that these interpretations come from automatically extracted visual features.

[Visual Assessment]
Write 2-4 concise sentences.
Describe visual properties inferred mainly from the image itself, such as:
- shading behavior
- grayscale or color appearance
- line-based vs filled appearance
- compact vs visually dispersed organization
- smooth vs sharp transitions
- apparent visual complexity
Remain conservative and avoid unsupported claims.

[Forensic Relevance]
Write 2-3 concise sentences.
Explain how the tattoo may contribute to forensic comparison as a soft biometric cue.
Focus on:
- visible structure
- composition
- contrast
- recognizable visual arrangement
Do NOT claim that the tattoo is unique, impossible to replicate, or a reliable identifier by itself.

[Limitations and Uncertainty]
Write 2-3 concise sentences.
Explicitly describe limitations and uncertainty.
Mention that:
- conclusions are based only on the visible crop and extracted features
- anatomical body location cannot be reliably inferred
- semantic meaning or personal significance cannot be reliably determined
- image quality, crop limitations, and visibility may affect interpretation
""".strip()

    return prompt

In [18]:
# Inference
def generate_report_qwen(crop_pil: Image.Image, prompt: str, max_new_tokens=500):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": crop_pil},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )

    inputs = inputs.to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    response = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    return response.strip()

In [19]:
# Save report
def save_report_txt(base_name: str, report: str):
    txt_path = os.path.join(BATCH_REPORTS_DIR, f"{base_name}_report.txt")

    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(report)

    return txt_path

In [ ]:
# Load features and select batch
all_features = read_json(FEATURES_JSON_PATH)

print("Total feature entries:", len(all_features))

start_idx = REPORT_BATCH_INDEX * REPORT_BATCH_SIZE
end_idx = min(start_idx + REPORT_BATCH_SIZE, len(all_features))

batch_features = all_features[start_idx:end_idx]

print(f"Processing report batch {REPORT_BATCH_INDEX}")
print(f"Range: {start_idx} - {end_idx - 1}")
print(f"Items in batch: {len(batch_features)}")

if not batch_features:
    raise RuntimeError("This report batch is empty. Lower REPORT_BATCH_INDEX.")

In [ ]:
# Process batch
errors = []
summary = []

for feat_json in tqdm(batch_features, desc=f"Generating reports batch {REPORT_BATCH_INDEX}"):
    filename = feat_json["metadata"]["filename"]
    base_name = os.path.splitext(filename)[0]

    try:
        image_path = os.path.join(IMAGE_DIR, filename)

        img_bgr = read_image_bgr(image_path)

        # No tattoo detected
        if no_tattoo_detected(feat_json):

          report = """
          ### [Observed Description]
          No visible tattoo region was detected in the provided image crop.
          """.strip()

        else:

          crop_bgr = crop_from_features(
                      img_bgr,
                      feat_json,
                      margin_ratio=MARGIN_RATIO
                      )

          crop_pil = bgr_to_pil(crop_bgr)

          prompt = build_forensic_prompt(feat_json)

          report = generate_report_qwen(
                    crop_pil,
                    prompt,
                    max_new_tokens=MAX_NEW_TOKENS
                    )

        txt_path = save_report_txt(base_name, report)

        summary.append({
            "filename": filename,
            "report_path": txt_path,
            "status": "ok"
        })

    except Exception as e:
        errors.append({
            "filename": filename,
            "error": str(e)
        })

        summary.append({
            "filename": filename,
            "report_path": None,
            "status": "error"
        })

In [22]:
# Save batch logs
with open(BATCH_ERRORS_OUT, "w", encoding="utf-8") as f:
    json.dump(errors, f, indent=2, ensure_ascii=False)

with open(BATCH_SUMMARY_OUT, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

In [ ]:
# Final summary
print("\n================ REPORT GENERATION SUMMARY ================\n")
print("Report batch:", REPORT_BATCH_INDEX)
print("Range:", start_idx, "-", end_idx - 1)
print("Reports generated:", len([x for x in summary if x["status"] == "ok"]))
print("Errors:", len(errors))
print("Reports folder:", BATCH_REPORTS_DIR)
print("Errors file:", BATCH_ERRORS_OUT)
print("Summary file:", BATCH_SUMMARY_OUT)